# Sales Prediction - Regression Modeling, 5-Fold CV & Feature Importance

This notebook covers:
1. **Data Preprocessing & Train/Test Split** (80/20 split, scaling via `StandardScaler`).
2. **Leakage-Free Pipelines** encapsulating `StandardScaler` and candidate regressors.
3. **5-Fold Cross-Validation** comparing:
   - Linear Regression
   - Random Forest Regressor
   - Gradient Boosting Regressor
4. **Holdout Test Set Evaluation** ($R^2$, MAE, RMSE).
5. **Actual vs. Predicted Visualizations**.
6. **Advertising Channel Feature Importance Analysis** (identifying dominant sales driver).
7. **Model Serialization** using `joblib`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.load_data import load_sales_dataframe

sns.set_theme(style="whitegrid")

## 1. Load Data & 80/20 Train-Test Split

In [ ]:
df = load_sales_dataframe(file_path="../data/advertising.csv")
feature_cols = ["tv", "radio", "newspaper"]
X = df[feature_cols]
y = df["sales"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Training set: {X_train.shape[0]} samples | Test set: {X_test.shape[0]} samples")

## 2. Define Pipelines & 5-Fold Cross-Validation

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), feature_cols)
])

models = {
    "Linear Regression": Pipeline([("preprocessor", preprocessor), ("regressor", LinearRegression())]),
    "Random Forest Regressor": Pipeline([("preprocessor", preprocessor), ("regressor", RandomForestRegressor(n_estimators=100, random_state=42))]),
    "Gradient Boosting Regressor": Pipeline([("preprocessor", preprocessor), ("regressor", GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42))])
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, pipeline in models.items():
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=["r2", "neg_mean_absolute_error", "neg_root_mean_squared_error"])
    cv_results[name] = {
        "CV R2": np.mean(scores["test_r2"]),
        "CV MAE": np.mean(-scores["test_neg_mae"]),
        "CV RMSE": np.mean(-scores["test_neg_root_mean_squared_error"])
    }
    print(f"{name:28s} | CV R2: {cv_results[name]['CV R2']:.4f} | CV MAE: {cv_results[name]['CV MAE']:.3f}k | CV RMSE: {cv_results[name]['CV RMSE']:.3f}k")

## 3. Holdout Test Set Evaluation

In [ ]:
test_results = {}
predictions = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    predictions[name] = y_pred
    
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    test_results[name] = {"R2": r2, "MAE": mae, "RMSE": rmse}
    print(f"{name:28s} | Test R2: {r2:.4f} | Test MAE: {mae:.3f}k | Test RMSE: {rmse:.3f}k")

## 4. Actual vs. Predicted Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ["#2980b9", "#27ae60", "#8e44ad"]

for idx, (name, y_pred) in enumerate(predictions.items()):
    axes[idx].scatter(y_test, y_pred, color=colors[idx], alpha=0.75, edgecolors="k")
    axes[idx].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", label="Perfect Fit (y=x)")
    axes[idx].set_title(f"{name}\n$R^2$: {test_results[name]['R2']:.3f} | RMSE: {test_results[name]['RMSE']:.2f}k")
    axes[idx].set_xlabel("Actual Sales (k Units)")
    axes[idx].set_ylabel("Predicted Sales (k Units)")
    axes[idx].legend()

plt.tight_layout()
plt.show()

## 5. Advertising Channel Feature Importance

In [ ]:
rf_model = models["Random Forest Regressor"].named_steps["regressor"]
imp_df = pd.DataFrame({"Channel": feature_cols, "Importance (%)": rf_model.feature_importances_ * 100})
imp_df.sort_values(by="Importance (%)", ascending=False, inplace=True)

plt.figure(figsize=(7, 4))
sns.barplot(data=imp_df, x="Channel", y="Importance (%)", palette="Greens_r")
plt.title("Channel Impact on Sales (Random Forest Importance)")
plt.ylabel("Importance (%)")
plt.show()

## 6. Save Best Model

In [ ]:
best_model_name = max(test_results.keys(), key=lambda k: test_results[k]["R2"])
best_pipeline = models[best_model_name]
models_dir = project_root / "models"
models_dir.mkdir(exist_ok=True)
joblib.dump({"model_name": best_model_name, "pipeline": best_pipeline, "feature_names": feature_cols}, models_dir / "best_model.joblib")
print(f"Serialized champion model: {best_model_name} to models/best_model.joblib")